0. 제출 정보
- 이름: 예민호
- GitHub ID: yegamanse-cpu
- 작성일: 2026. 10. 19
- 최종 Notebook URL: https://github.com/yegamanse-cpu/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb

In [41]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()


def find_project_root(start: Path) -> Path:
    """src 폴더(또는 requirements.txt)가 있는 상위 폴더를 프로젝트 루트로 찾는다."""
    for candidate in [start, *start.parents]:
        if (candidate / 'src').is_dir() or (candidate / 'requirements.txt').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(CURRENT_DIR)   # src 모듈 import 용 (sys.path 추가)
NOTEBOOK_DIR = CURRENT_DIR                        # 이 노트북(ch06) 폴더

# 전처리 데이터는 ch06 사본을 사용한다. (사본이 없으면 프로젝트 루트의 data/processed 로 대체)
PROCESSED_DIR = NOTEBOOK_DIR / 'data' / 'processed'
if not PROCESSED_DIR.exists():
    PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

REPORT_DIR = NOTEBOOK_DIR / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)

현재 실행 위치: c:\dev\llm-data-analysis-course\notebooks\ch06
프로젝트 루트: c:\dev\llm-data-analysis-course
전처리 데이터 폴더: c:\dev\llm-data-analysis-course\notebooks\ch06\data\processed
보고서 폴더: c:\dev\llm-data-analysis-course\notebooks\ch06\reports


- 과제 후반 스크립트 실행 중 src모듈이 불러오기가 되지 않아 llm의 도움을 받아 코드를 재수정
- 원인은 src모듈이 루트폴더에 위치하고 있으나 위치를 잡지 못해 오류가 난것으로 판단됨

In [42]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')

전처리 데이터 불러오기 완료


- 이 단계에서 전처리 데이터가 c:\dev\llm-data-analysis-course\notebooks\ch05\data\processed 폴더에 있어 데이터 불러오기 오류 발생
- ch05/data/...에 있는 전처리 데이터를 ch06/data/...에 복사하여 오류 해결함

In [43]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()

[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (96, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (752, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [44]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())

order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


In [45]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions

,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


In [46]:
customers['age'].describe()

count    150.000000
mean      43.913333
std       17.802397
min       18.000000
25%       30.000000
50%       41.000000
75%       56.750000
max      150.000000
Name: age, dtype: float64

In [47]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city

,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [48]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender

,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


In [49]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category

,category,product_count,product_ratio
0,패션,16,16.67
1,뷰티,16,16.67
2,식품,16,16.67
3,도서,13,13.54
4,전자기기,13,13.54
5,스포츠,10,10.42
6,생활용품,9,9.38
7,전자 기기,1,1.04
8,생활 용품,1,1.04
9,뷰 티,1,1.04


In [50]:
products['price'].describe()

count    9.600000e+01
mean     2.048958e+05
std      1.011848e+06
min      5.000000e+03
25%      5.275000e+04
50%      9.450000e+04
75%      1.532500e+05
max      9.999999e+06
Name: price, dtype: float64

In [51]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price

,category,product_count,avg_price,min_price,max_price
2,뷰티,16,710937.0,18000.0,9999999.0
3,생활 용품,1,124000.0,124000.0,124000.0
6,식품,16,120812.0,11000.0,195000.0
8,전자기기,13,120231.0,38000.0,178000.0
9,패션,16,115125.0,28000.0,197000.0
0,도서,13,103692.0,34000.0,182000.0
5,스포츠,10,78400.0,5000.0,183000.0
4,생활용품,9,69556.0,29000.0,147000.0
1,뷰 티,1,54000.0,54000.0,54000.0
7,전자 기기,1,21000.0,21000.0,21000.0


In [52]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status

,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


In [ ]:
if 'order_sales' not in globals():
    order_sales = order_items.merge(orders, on='order_id', how='left')
    order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
    order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

completed_sales = order_sales[order_sales['order_status'] == 'completed']

completed_order_count = completed_sales['order_id'].nunique()
completed_total = completed_sales['line_total'].sum()
overall_total = order_sales['line_total'].sum()
completed_ratio = round(completed_total / overall_total * 100, 1)

print(f'completed 주문 수: {completed_order_count:,}건')
print(f'completed 매출 합계: {completed_total:,.0f}원')
print(f'전체 매출 합계: {overall_total:,.0f}원')
print(f'completed 매출 비중: {completed_ratio}%')

completed_by_category = (
    completed_sales
    .merge(products[['product_id', 'category']], on='product_id', how='left')
    .groupby('category', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)
completed_by_category['sales_ratio'] = (
    completed_by_category['total_sales'] / completed_by_category['total_sales'].sum() * 100
).round(2)

completed_by_category

completed 주문 수: 179건
completed 매출 합계: 141,340,000원
전체 매출 합계: 252,140,000원
completed 매출 비중: 56.1%


,category,order_count,total_quantity,total_sales,sales_ratio
9,패션,68,290.0,32940000.0,23.99
6,식품,63,226.0,25035000.0,18.23
8,전자기기,54,176.0,21485000.0,15.65
0,도서,49,174.0,17663000.0,12.86
2,뷰티,54,168.0,17103000.0,12.45
5,스포츠,47,165.0,13237000.0,9.64
4,생활용품,37,108.0,7001000.0,5.10
3,생활 용품,5,17.0,2108000.0,1.54
1,뷰 티,4,12.0,648000.0,0.47
7,전자 기기,1,5.0,105000.0,0.08


- 교수님이 제공해주신 가이드 노트북에 completed 주문만 집계하는 코드가 없어 llm으로 코드를 생성하여 확인

In [54]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method

,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [55]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())

주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00
주문 월 개수: 16


In [56]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()

,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [57]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (752, 6)
병합 후 sales_items: (752, 9)
상품명 누락: 26
카테고리 누락: 26


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,39,2.0,172000.0,344000.0,뷰티 상품 039,뷰티,172000.0
1,2,2,26,4.0,79000.0,316000.0,뷰티 상품 026,뷰티,79000.0
2,3,2,46,3.0,87000.0,261000.0,생활용품 상품 046,생활용품,87000.0
3,4,2,80,1.0,187000.0,187000.0,패션 상품 080,패션,187000.0
4,5,2,91,5.0,125000.0,625000.0,전자기기 상품 091,전자기기,125000.0


In [58]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales

,category,total_quantity,total_sales,sales_ratio
9,패션,545.0,64890000.0,26.39
6,식품,394.0,47044000.0,19.13
8,전자기기,303.0,37542000.0,15.27
2,뷰티,282.0,28614000.0,11.64
0,도서,277.0,28246000.0,11.49
5,스포츠,254.0,20651000.0,8.40
4,생활용품,213.0,14303000.0,5.82
3,생활 용품,27.0,3348000.0,1.36
1,뷰 티,19.0,1026000.0,0.42
7,전자 기기,12.0,252000.0,0.10


In [59]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price

,category,total_quantity,total_sales,sales_ratio,avg_price
0,패션,545.0,64890000.0,26.39,115125.0
1,식품,394.0,47044000.0,19.13,120812.0
2,전자기기,303.0,37542000.0,15.27,120231.0
3,뷰티,282.0,28614000.0,11.64,710937.0
4,도서,277.0,28246000.0,11.49,103692.0
5,스포츠,254.0,20651000.0,8.40,78400.0
6,생활용품,213.0,14303000.0,5.82,69556.0
7,생활 용품,27.0,3348000.0,1.36,124000.0
8,뷰 티,19.0,1026000.0,0.42,54000.0
9,전자 기기,12.0,252000.0,0.10,21000.0


In [60]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()

병합 후 order_sales: (752, 12)
order_date 누락: 11


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,39,2.0,172000.0,344000.0,144.0,2026-07-12,naver_pay,completed,2026-07,Sunday
1,2,2,26,4.0,79000.0,316000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
2,3,2,46,3.0,87000.0,261000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
3,4,2,80,1.0,187000.0,187000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
4,5,2,91,5.0,125000.0,625000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday


In [61]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales

,order_month,total_sales,order_count,avg_order_value
0,2025-07,13729000.0,19,722579.0
1,2025-08,32639000.0,25,1305560.0
2,2025-09,13058000.0,12,1088167.0
3,2025-10,16680000.0,22,758182.0
4,2025-11,15080000.0,23,655652.0
5,2025-12,16152000.0,23,702261.0
6,2026-01,19053000.0,26,732808.0
7,2026-02,14207000.0,19,747737.0
8,2026-03,13841000.0,22,629136.0
9,2026-04,21007000.0,26,807962.0


In [62]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)

,customer_id,name,city,order_count,total_sales,avg_order_value
15,18.0,서하현,대전,2,16666000.0,8333000.0
2,3.0,이예우,서울,7,8909000.0,1272714.0
17,20.0,김성희,성남,6,6200000.0,1033333.0
45,52.0,박유현,울산,5,5388000.0,1077600.0
11,14.0,임서영,대구,4,5204000.0,1301000.0
19,22.0,강영민,부산,4,5029000.0,1257250.0
48,55.0,장유미,부산,6,4850000.0,808333.0
74,90.0,윤서미,수원,5,4847000.0,969400.0
12,15.0,강수영,광주,5,4281000.0,856200.0
77,93.0,윤도우,울산,4,4001000.0,1000250.0


In [63]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary

,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


In [64]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))

[WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch06/reports/ch06_category_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch06/reports/ch06_customer_city.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch06/reports/ch06_customer_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch06/reports/ch06_eda_questions.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch06/reports/ch06_monthly_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch06/reports/ch06_product_category.csv')]

In [65]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)

EDA 요약 보고서 저장 완료: c:\dev\llm-data-analysis-course\notebooks\ch06\reports\ch06_eda_summary.md


In [66]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()

dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [67]:
module_results['category_sales'].head()

,category,total_quantity,total_sales,sales_ratio
9,패션,545.0,64890000.0,26.39
6,식품,394.0,47044000.0,19.13
8,전자기기,303.0,37542000.0,15.27
2,뷰티,282.0,28614000.0,11.64
0,도서,277.0,28246000.0,11.49


1. 내가 정한 EDA 질문

| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 고객은 어떤 지역·연령대에 분포하는가? | 고객 | 고객 수, 평균 나이 | `describe` + `value_counts`, 비율 합 100% 확인 |
| 어떤 카테고리의 상품이 많은가? | 상품 | 카테고리별 상품 수 | `value_counts`, 같은 카테고리가 다른 표기로 쪼개졌는지 점검 |
| 주문 상태·결제수단 분포는 어떤가? | 주문 | 주문 수, 비율 | `value_counts`, 합계가 전체 주문 수(300)와 맞는지 대조 |
| 카테고리별 매출은 어떻게 다른가? | 매출 | 총매출, 매출 비중 | `groupby`합계, 카테고리 합 vs 전체 `line_total` 합 대조 |
| 월별 매출·주문 수는 어떻게 변하는가? | 시간 | 월별 매출, 월별 주문 수 |`order_month` 기준 `grupby`, 월별 합 vs 전체 합 대조 |
| 구매 금액이 높은 고객은 누구인가? | 고객 가치 | 고객별 총매출, 주문 횟수 | `customer_id` 기준 `groupby`, 총매출 내림차순 상위 확인 |

- 왜 이 질문을 선택했는가? : 챕터05에서 전처리한 자료를 활용하여 추가적 작업을 최소화한 질문 선택


2. 핵심 집계 결과

- completed 원본 합계: 141,340,000원 (전체 252,140,000원 대비 56.1%. 나머지 cancelled 21% + refunded 19%)

- 카테고리별 합계: 245,916,000원 (패션 64,890,000원 26.39% ~ 전자 기기 252,000원, 결측 6,224,000원 빠짐)

- 월별 합계: 248,630,000원 (order_date 결측 3,510,000원 제외)

- 총합 일치 여부: 불일치 (전체 252,140,000, 카테고리 245,916,000, 월별 248,630,000)

![01 캡처](images/01.png)

- 결과 관찰 : 같은 line_total을 세 방식으로 더했는데 합이 다 다른 것을 확인

- 나의 해석과 판단 : 계산 실수가 아닌 조인 시 행이 탈락한 결과로 매칭되지 않은 행은 결측이 되고, groupby에서 빠지면서 부분 합이 전체보다 작아진 것으로 판단

- 업무·분석적 의미 : 부분 합과 전체 합을 대조하지 않았다면 결측이 발생한 것을 모르기 때문에 집계 시 항상 "빠진 행이 없는지" 확인하는 대조 필요

- 한계와 추가 확인 사항 : 결측 날짜 11건과 카테고리 결측 26건이 동일한 주문에서 겹치는지 확인하지 못함

3. 관찰 → 가설 → 추가 검증

사례 1
- 관찰: age의 max 150으로 불가능한 값이 확인되며 products도 price max가 9,999,999원으로 뷰티 평균단가 이상발생
- 가설: 기록 오류로 150, 9999999 같은 실제 값이 아닌 숫자일 가능성 있음
- 추가 검증: age 100 넘는 값과 price 상위 값을 직접 뽑아 확인

사례 2
- 관찰: 집계에서 '전자기기'와 '전자 기기', '생활용품'과 '생활 용품', '뷰티'와 '뷰 티'가 따로 잡힘
- 가설: 전처리 과정 중 공백 정규화가 빠진 것으로 보임
- 추가 검증: 공백 제거 후 다시 카테고리가 7개로 주어드는지 확인

사례 3
- 관찰: 주문 종료일이 2028-12-31까지 있으며 order_date 변환 실패 3건
- 가설: 분석구간 밖의 데이터로 판단
- 추가 검증: 2026-08 이후 주문을 점검

In [70]:
age_outlier = customers[customers['age'] > 100]
print('age 100 초과 고객 수:', len(age_outlier))
print(age_outlier[['customer_id', 'name', 'age']].to_string(index=False))

print()
print('price 상위 5개 상품:')
print(
    products.sort_values('price', ascending=False)
    .head(5)[['product_id', 'product_name', 'category', 'price']]
    .to_string(index=False)
)

age 100 초과 고객 수: 1
 customer_id name   age
          33  권승윤 150.0

price 상위 5개 상품:
 product_id product_name category     price
         99    뷰티 상품 099       뷰티 9999999.0
         23    패션 상품 023       패션  197000.0
         72    식품 상품 072       식품  195000.0
         36    뷰티 상품 036       뷰티  193000.0
         65    식품 상품 065       식품  193000.0


In [ ]:
products_norm = products.copy()
products_norm['category'] = products_norm['category'].str.replace(' ', '', regex=False)

print('공백 정규화 전 카테고리 수:', products['category'].nunique())
print('공백 정규화 후 카테고리 수:', products_norm['category'].nunique())

category_sales_norm = (
    order_items
    .merge(products_norm[['product_id', 'category']], on='product_id', how='left')
    .groupby('category', as_index=False)
    .agg(total_sales=('line_total', 'sum'))
    .sort_values('total_sales', ascending=False)
)
category_sales_norm['sales_ratio'] = (
    category_sales_norm['total_sales'] / category_sales_norm['total_sales'].sum() * 100
).round(2)

category_sales_norm

공백 정규화 전 카테고리 수: 10
공백 정규화 후 카테고리 수: 7


,category,total_sales,sales_ratio
6,패션,64890000.0,26.39
4,식품,47044000.0,19.13
5,전자기기,37794000.0,15.37
1,뷰티,29640000.0,12.05
0,도서,28246000.0,11.49
3,스포츠,20651000.0,8.40
2,생활용품,17651000.0,7.18


In [71]:
orders_date = orders.copy()
orders_date['order_date'] = pd.to_datetime(orders_date['order_date'], errors='coerce')

future_orders = orders_date[orders_date['order_date'] > '2026-08-31']
missing_date = orders_date[orders_date['order_date'].isna()]

print('2026-08 이후(미래) 주문 수:', len(future_orders))
print(future_orders[['order_id', 'customer_id', 'order_date', 'order_status']].to_string(index=False))

print()
print('order_date 결측 주문 수:', len(missing_date))
print(missing_date[['order_id', 'customer_id', 'order_date', 'order_status']].to_string(index=False))

2026-08 이후(미래) 주문 수: 2
 order_id  customer_id order_date order_status
      256           55 2027-01-15    completed
      287           83 2028-12-31    cancelled

order_date 결측 주문 수: 3
 order_id  customer_id order_date order_status
       72          125        NaT    completed
      143           19        NaT     refunded
      214           62        NaT    completed


## 4. LLM 질문 확장

- Safe Context: 개인정보 빼고 집계표만 전달. 카테고리별 매출, 월별 매출, 주문 상태 비율, 데이터 품질 제공
- Prompt: 위 EDA 요약을 보고, 지금 데이터로 바로 확인할 수 있는 추가 분석 질문 3개만 제안해줘
- LLM 제안 요약:

    카테고리별 매출 차이는 판매수량 때문인가, 상품 단가 때문인가

    월별 매출 변동은 어떤 카테고리가 주도했는가?

    고액 구매 고객은 반복 구매형인가, 일회성 고액 구매형인가?
- 실제 사용 여부: 사용
- 판단 이유: 세 개 다 현재 컬럼으로 확인 가능하므로 사용
- 사람이 수정한 내용: 없음

![02 캡처](images/02.png)


5. 다음 분석 우선순위

    카테고리 공백 통일, age 150, price 999만원 등 이상치 처리, 미래 날짜 주문 어떻게 할지 결정

    취소·환불이 특정 카테고리·결제수단·고객에 몰리는지 확인

    고액 고객이 반복 구매인지 한 번에 크게 산 건지 점검

- 가장 먼저 확인하고 싶은 이유 : 전처리 및 결측, 이상치 처리를 정리안하면 집계가 모두 흔들리기 때문

- 현재 데이터만으로 단정할 수 없는 것 : 미래 날짜가 입력오류인지 실제 데이터인지 판단 불가

## 최종 체크
- [x] 질문과 지표가 연결되어 있습니다.
- [x] 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 다음 검증 질문을 작성했습니다.
- [x] LLM 제안을 검증했습니다.
- [x] 최종 Notebook URL을 제출합니다.